Colab notebook: https://colab.research.google.com/drive/1tZVdOYvKpU7uM92-qyJzMIBcrV77aiqP?usp=sharing

# SMD Colab Template: Loading, Preprocessing, and Windowing

This notebook is a reusable Colab template for quick time-series experiments on the Server Machine Dataset (SMD).

It follows the repository preference of readability-first research code:

- Keep dataset parsing, standardization, windowing, transforms, and model adapters separate.
- Fit preprocessing statistics on training data only.
- Use an explicit batch contract: `x`, `point_labels`, `mask`, `timestamps`, and `meta`.
- Keep cells small enough to inspect and rerun line-by-line.

Expected Colab dataset location:

```text
/content/ServerMachineDataset/
|-- train/
|-- test/
|-- test_label/
`-- interpretation_label/
```


## 1. Imports

Run this cell first. It only imports common packages that are already available in Colab.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


## 2. Reproducibility

Keep this in a separate cell so you can change the seed without touching the data code.

In [ ]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 3. Experiment Configuration

For quick experiments, start with one machine. Change `SELECTED_MACHINE_IDS` to `"all"` when you want all 28 SMD machines.

In [ ]:
DATA_ROOT = Path("/content/ServerMachineDataset")

SELECTED_MACHINE_IDS: list[str] | str = ["machine-1-1"]
# SELECTED_MACHINE_IDS = "all"

VALIDATION_SPLIT_RATIO = 0.2
WINDOW_SIZE = 100
STRIDE = 10
BATCH_SIZE = 32
NUM_WORKERS = 0

# Use small limits for fast Colab iteration. Set any value to None for the full split.
MAX_WINDOWS_BY_SPLIT = {
    "train": 2048,
    "val": 512,
    "test": None,
}


## 3.1 Download SMD from GitHub

Run this section only when `/content/ServerMachineDataset` is missing or incomplete. The code follows the downloader mechanism from `notebooks/time_series_loading_template.ipynb`, but writes directly to `DATA_ROOT` so the preprocessing cells below do not need to change.

In [ ]:
try:
    import requests
except ImportError as error:
    raise ImportError("Install requests first with `!pip -q install requests`, then rerun this cell.") from error


In [ ]:
DATASET_REPOSITORY_OWNER = "NetManAIOps"
DATASET_REPOSITORY_NAME = "OmniAnomaly"
DATASET_REPOSITORY_BRANCH = "master"
DATASET_DIRECTORY_IN_REPOSITORY = "ServerMachineDataset"

LOCAL_OUTPUT_DIRECTORY = DATA_ROOT


In [ ]:
REQUIRED_DATASET_DIRECTORIES = ["train", "test", "test_label"]
OPTIONAL_DATASET_CHILDREN = ["LICENSE", "interpretation_label"]
EXPECTED_TOP_LEVEL_CHILDREN = set(REQUIRED_DATASET_DIRECTORIES + OPTIONAL_DATASET_CHILDREN)


In [ ]:
http_session = requests.Session()
http_session.headers.update(
    {
        "Accept": "application/vnd.github+json",
        "User-Agent": "smd-dataset-downloader",
    }
)

http_session.headers


In [ ]:
def build_github_contents_api_url(path_in_repository: str) -> str:
    return (
        f"https://api.github.com/repos/"
        f"{DATASET_REPOSITORY_OWNER}/"
        f"{DATASET_REPOSITORY_NAME}/contents/"
        f"{path_in_repository}"
        f"?ref={DATASET_REPOSITORY_BRANCH}"
    )


In [ ]:
def request_github_json(path_in_repository: str) -> list[dict[str, Any]]:
    api_url = build_github_contents_api_url(path_in_repository)
    response = http_session.get(api_url, timeout=30)
    response.raise_for_status()

    github_items = response.json()
    if not isinstance(github_items, list):
        raise TypeError(f"Expected a directory listing for {path_in_repository}, got {type(github_items).__name__}")
    return github_items


In [ ]:
dataset_directories_present = all((DATA_ROOT / directory_name).exists() for directory_name in REQUIRED_DATASET_DIRECTORIES)

if dataset_directories_present:
    top_level_items = []
    print(f"Dataset directories already exist under {DATA_ROOT}; skipping GitHub metadata request.")
else:
    top_level_items = request_github_json(DATASET_DIRECTORY_IN_REPOSITORY)

    for item in top_level_items:
        print(
            "name =", item["name"],
            "| type =", item["type"],
            "| path =", item["path"],
        )


In [ ]:
def download_binary_file(file_download_url: str, local_output_file_path: Path) -> None:
    local_output_file_path.parent.mkdir(parents=True, exist_ok=True)

    file_response = http_session.get(file_download_url, timeout=60)
    file_response.raise_for_status()

    local_output_file_path.write_bytes(file_response.content)
    print(f"Saved file: {local_output_file_path}")


In [ ]:
def recursively_download_github_directory_verbose(
    directory_path_in_repository: str,
    local_output_directory: Path,
) -> None:
    github_items = request_github_json(directory_path_in_repository)

    print(f"Traversing repository path: {directory_path_in_repository}")
    print(f"Local output directory: {local_output_directory}")

    local_output_directory.mkdir(parents=True, exist_ok=True)

    for github_item in github_items:
        github_item_type = github_item["type"]
        github_item_name = github_item["name"]
        github_item_path = github_item["path"]

        print("Handling:", "| type =", github_item_type, "| name =", github_item_name, "| path =", github_item_path)

        if github_item_type == "dir":
            recursively_download_github_directory_verbose(
                directory_path_in_repository=github_item_path,
                local_output_directory=local_output_directory / github_item_name,
            )
        elif github_item_type == "file":
            download_binary_file(
                file_download_url=github_item["download_url"],
                local_output_file_path=local_output_directory / github_item_name,
            )
        else:
            raise ValueError(f"Unsupported GitHub item type: {github_item_type} for path {github_item_path}")


In [ ]:
dataset_directories_present = all((DATA_ROOT / directory_name).exists() for directory_name in REQUIRED_DATASET_DIRECTORIES)

if dataset_directories_present:
    print(f"Skipping download because dataset directories already exist under {DATA_ROOT}.")
else:
    recursively_download_github_directory_verbose(
        directory_path_in_repository=DATASET_DIRECTORY_IN_REPOSITORY,
        local_output_directory=LOCAL_OUTPUT_DIRECTORY,
    )


In [ ]:
local_top_level_children = {path.name for path in DATA_ROOT.iterdir()}
missing_required_directories = sorted(set(REQUIRED_DATASET_DIRECTORIES) - local_top_level_children)
missing_optional_children = sorted(set(OPTIONAL_DATASET_CHILDREN) - local_top_level_children)

print("local top-level children =", sorted(local_top_level_children))
print("missing required directories =", missing_required_directories)
print("missing optional children =", missing_optional_children)

if missing_required_directories:
    raise FileNotFoundError(f"SMD preprocessing directories are missing under {DATA_ROOT}: {missing_required_directories}")


## 4. Inspect the Downloaded SMD Folder

This checks that the files were already downloaded into `/content/ServerMachineDataset` before we load anything.

In [ ]:
expected_subdirectories = ["train", "test", "test_label"]

for subdirectory_name in expected_subdirectories:
    subdirectory_path = DATA_ROOT / subdirectory_name
    print(subdirectory_path, "exists =", subdirectory_path.exists())
    print("number of txt files =", len(list(subdirectory_path.glob("*.txt"))) if subdirectory_path.exists() else 0)


## 5. List Machine IDs

The machine ID comes from the file stem, for example `machine-1-1` from `machine-1-1.txt`.

In [ ]:
def list_smd_machine_ids(data_root: Path) -> list[str]:
    train_directory = data_root / "train"
    machine_ids = [file_path.stem for file_path in train_directory.glob("*.txt")]
    return sorted(machine_ids)


In [ ]:
all_machine_ids = list_smd_machine_ids(DATA_ROOT)

print("number of machine ids =", len(all_machine_ids))
print("first five machine ids =", all_machine_ids[:5])


## 6. Select Machines for This Run

This keeps the selection explicit and easy to modify for ablation-style experiments.

In [ ]:
if SELECTED_MACHINE_IDS == "all":
    selected_machine_ids = all_machine_ids
else:
    selected_machine_ids = list(SELECTED_MACHINE_IDS)

missing_machine_ids = sorted(set(selected_machine_ids) - set(all_machine_ids))
if missing_machine_ids:
    raise ValueError(f"Unknown SMD machine ids: {missing_machine_ids}")

print("selected machine ids =", selected_machine_ids)
print("number of selected machines =", len(selected_machine_ids))


## 7. Raw File Readers

SMD feature files are comma-separated matrices with shape `[time, channels]`.

SMD test label files are point labels with shape `[time]`, where `1` marks anomalous time points.

In [ ]:
def load_smd_feature_matrix(file_path: Path) -> np.ndarray:
    feature_matrix = np.loadtxt(file_path, delimiter=",", dtype=np.float32)
    if feature_matrix.ndim != 2:
        raise ValueError(f"Expected a 2D feature matrix at {file_path}, got shape {feature_matrix.shape}")
    return feature_matrix


In [ ]:
def load_smd_label_vector(file_path: Path) -> np.ndarray:
    label_vector = np.loadtxt(file_path, delimiter=",", dtype=np.float32)
    label_vector = label_vector.astype(np.int64)
    if label_vector.ndim != 1:
        raise ValueError(f"Expected a 1D label vector at {file_path}, got shape {label_vector.shape}")
    return label_vector


## 8. Raw Sequence Contract

Each raw sequence uses the same keys as the project data contract:

```python
sequence = {
    "x": array[time, channels],
    "point_labels": array[time] or None,
    "mask": None,
    "timestamps": None,
    "meta": dict,
}
```

In [ ]:
def build_raw_sequence(
    x_array: np.ndarray,
    point_labels: np.ndarray | None,
    machine_id: str,
    split_name: str,
) -> dict[str, Any]:
    if point_labels is not None and len(point_labels) != len(x_array):
        raise ValueError(f"Label length does not match sequence length for {machine_id} {split_name}")

    return {
        "x": x_array.astype(np.float32),
        "point_labels": None if point_labels is None else point_labels.astype(np.int64),
        "mask": None,
        "timestamps": None,
        "meta": {
            "dataset_name": "smd",
            "entity_id": machine_id,
            "split": split_name,
            "num_channels": int(x_array.shape[1]),
            "sequence_length": int(x_array.shape[0]),
        },
    }


## 9. Load One Machine and Create a Validation Split

SMD provides train and test files. This template splits the training file into train and validation slices, then uses the provided test labels for the test split.

In [ ]:
def load_one_smd_machine(
    data_root: Path,
    machine_id: str,
    validation_split_ratio: float,
) -> dict[str, dict[str, Any]]:
    train_array = load_smd_feature_matrix(data_root / "train" / f"{machine_id}.txt")
    test_array = load_smd_feature_matrix(data_root / "test" / f"{machine_id}.txt")
    test_labels = load_smd_label_vector(data_root / "test_label" / f"{machine_id}.txt")

    validation_length = max(1, int(len(train_array) * validation_split_ratio))
    train_cutoff = len(train_array) - validation_length
    if train_cutoff < 1:
        raise ValueError(f"Validation split leaves no training points for {machine_id}")

    train_labels = np.zeros(train_cutoff, dtype=np.int64)
    validation_labels = np.zeros(validation_length, dtype=np.int64)

    return {
        "train": build_raw_sequence(train_array[:train_cutoff], train_labels, machine_id, "train"),
        "val": build_raw_sequence(train_array[train_cutoff:], validation_labels, machine_id, "val"),
        "test": build_raw_sequence(test_array, test_labels, machine_id, "test"),
    }


## 10. Load the Selected Machines

The output keeps the split structure explicit: `raw_sequences_by_split["train"]`, `raw_sequences_by_split["val"]`, and `raw_sequences_by_split["test"]`.

In [ ]:
raw_sequences_by_split = {"train": [], "val": [], "test": []}

for machine_id in selected_machine_ids:
    machine_sequences = load_one_smd_machine(DATA_ROOT, machine_id, VALIDATION_SPLIT_RATIO)
    for split_name in raw_sequences_by_split:
        raw_sequences_by_split[split_name].append(machine_sequences[split_name])

for split_name, split_sequences in raw_sequences_by_split.items():
    print(split_name, "number of sequences =", len(split_sequences))
    print(split_name, "first sequence shape =", split_sequences[0]["x"].shape)


## 11. Fit Standardization on Training Data Only

This prevents leakage from validation and test data into preprocessing statistics.

In [ ]:
def fit_standard_scaler(train_sequences: list[dict[str, Any]], epsilon: float = 1e-6) -> dict[str, np.ndarray | float]:
    all_training_points = np.concatenate([sequence["x"] for sequence in train_sequences], axis=0)
    feature_mean = all_training_points.mean(axis=0).astype(np.float32)
    feature_std = all_training_points.std(axis=0).astype(np.float32)
    feature_std = np.maximum(feature_std, epsilon).astype(np.float32)

    return {
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "epsilon": float(epsilon),
    }


In [ ]:
def transform_sequence_with_standard_scaler(
    sequence: dict[str, Any],
    scaler_state: dict[str, np.ndarray | float],
) -> dict[str, Any]:
    transformed_sequence = dict(sequence)
    transformed_sequence["x"] = (
        (sequence["x"] - scaler_state["feature_mean"]) / scaler_state["feature_std"]
    ).astype(np.float32)
    transformed_sequence["meta"] = dict(sequence["meta"])
    transformed_sequence["meta"]["preprocessing"] = "standardized_with_train_split_statistics"
    return transformed_sequence


In [ ]:
standard_scaler_state = fit_standard_scaler(raw_sequences_by_split["train"])

scaled_sequences_by_split = {
    split_name: [
        transform_sequence_with_standard_scaler(sequence, standard_scaler_state)
        for sequence in split_sequences
    ]
    for split_name, split_sequences in raw_sequences_by_split.items()
}

print("feature mean shape =", standard_scaler_state["feature_mean"].shape)
print("feature std shape =", standard_scaler_state["feature_std"].shape)


## 12. Visualize a Standardized Sequence

Plot only a few channels at first. SMD has 38 channels, and plotting all channels usually hides the useful shape.

In [ ]:
sequence_to_plot = scaled_sequences_by_split["train"][0]
number_of_channels_to_plot = min(5, sequence_to_plot["x"].shape[1])

plt.figure(figsize=(14, 4))
plt.plot(sequence_to_plot["x"][:500, :number_of_channels_to_plot])
plt.title(f"Standardized training sequence: {sequence_to_plot['meta']['entity_id']}")
plt.xlabel("time index")
plt.ylabel("standardized value")
plt.show()


## 13. Window Dataset

The dataset stores only `(sequence_index, start_index, end_index)` records. It materializes windows on demand, which is easier to inspect and avoids copying every window upfront.

In [ ]:
class SMDWindowDataset(Dataset):
    def __init__(
        self,
        sequences: list[dict[str, Any]],
        window_size: int,
        stride: int,
        max_windows: int | None = None,
    ) -> None:
        self.sequences = sequences
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.index_records: list[tuple[int, int, int]] = []

        for sequence_index, sequence in enumerate(sequences):
            sequence_length = int(sequence["x"].shape[0])
            if sequence_length < self.window_size:
                continue

            for start_index in range(0, sequence_length - self.window_size + 1, self.stride):
                end_index = start_index + self.window_size
                self.index_records.append((sequence_index, start_index, end_index))

                if max_windows is not None and len(self.index_records) >= max_windows:
                    return

    def __len__(self) -> int:
        return len(self.index_records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        sequence_index, start_index, end_index = self.index_records[index]
        sequence = self.sequences[sequence_index]

        return {
            "x": sequence["x"][start_index:end_index].copy(),
            "point_labels": None if sequence["point_labels"] is None else sequence["point_labels"][start_index:end_index].copy(),
            "mask": None,
            "timestamps": None,
            "meta": {
                "dataset_name": sequence["meta"]["dataset_name"],
                "entity_id": sequence["meta"]["entity_id"],
                "split": sequence["meta"]["split"],
                "start_index": int(start_index),
                "end_index": int(end_index),
                "window_size": int(self.window_size),
            },
        }


## 14. Create Window Datasets

Start with limited windows for fast experiments. Increase or remove the limits later.

In [ ]:
window_datasets = {
    split_name: SMDWindowDataset(
        sequences=split_sequences,
        window_size=WINDOW_SIZE,
        stride=STRIDE,
        max_windows=MAX_WINDOWS_BY_SPLIT.get(split_name),
    )
    for split_name, split_sequences in scaled_sequences_by_split.items()
}

for split_name, dataset in window_datasets.items():
    print(split_name, "number of windows =", len(dataset))


## 15. Inspect One Window Before Batching

This is useful for checking metadata and label alignment before the DataLoader hides individual samples inside batches.

In [ ]:
first_train_window = window_datasets["train"][0]

print("x shape =", first_train_window["x"].shape)
print("point label shape =", first_train_window["point_labels"].shape)
print("meta =", first_train_window["meta"])


## 16. Collate Windows into the Project Batch Contract

The batch uses `x` with shape `[batch, window, channels]` and `point_labels` with shape `[batch, window]`.

In [ ]:
def collate_smd_windows(windows: list[dict[str, Any]]) -> dict[str, Any]:
    batch_x = torch.from_numpy(np.stack([window["x"] for window in windows], axis=0)).float()
    batch_point_labels = torch.from_numpy(
        np.stack([window["point_labels"] for window in windows], axis=0)
    ).long()

    return {
        "x": batch_x,
        "point_labels": batch_point_labels,
        "mask": None,
        "timestamps": None,
        "meta": [dict(window["meta"]) for window in windows],
    }


## 17. DataLoaders

Training can shuffle windows. Validation and test stay ordered so plots and error analysis are easier to trace back to machine IDs and time indices.

In [ ]:
data_loaders = {
    "train": DataLoader(
        window_datasets["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_smd_windows,
    ),
    "val": DataLoader(
        window_datasets["val"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_smd_windows,
    ),
    "test": DataLoader(
        window_datasets["test"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_smd_windows,
    ),
}


In [ ]:
sample_batch = next(iter(data_loaders["train"]))

print("x shape =", tuple(sample_batch["x"].shape))
print("point_labels shape =", tuple(sample_batch["point_labels"].shape))
print("first meta record =", sample_batch["meta"][0])

assert sample_batch["x"].ndim == 3
assert sample_batch["point_labels"].shape == sample_batch["x"].shape[:2]


## 18. Convert Point Labels to Window Labels

Many baselines need one label per window. A window is anomalous if any point inside it is anomalous.

In [ ]:
def point_labels_to_window_labels(point_labels: torch.Tensor) -> torch.Tensor:
    return (point_labels.sum(dim=1) > 0).long()


In [ ]:
sample_window_labels = point_labels_to_window_labels(sample_batch["point_labels"])

print("window labels shape =", tuple(sample_window_labels.shape))
print("number of anomalous windows in sample batch =", int(sample_window_labels.sum().item()))


## 19. Plot a Window

This is the first sanity check before fitting any model.

In [ ]:
window_index = 0
number_of_channels_to_plot = min(5, sample_batch["x"].shape[-1])

plt.figure(figsize=(14, 4))
plt.plot(sample_batch["x"][window_index, :, :number_of_channels_to_plot].cpu().numpy())
plt.title(f"Sample train window, label = {int(sample_window_labels[window_index].item())}")
plt.xlabel("window time index")
plt.ylabel("standardized value")
plt.show()


## 20. Find and Plot an Anomalous Test Window

This cell helps you quickly inspect whether labels line up with visible changes in the signal.

In [ ]:
def find_first_batch_with_anomaly(data_loader: DataLoader) -> dict[str, Any] | None:
    for batch in data_loader:
        if batch["point_labels"].sum().item() > 0:
            return batch
    return None


In [ ]:
anomalous_test_batch = find_first_batch_with_anomaly(data_loaders["test"])

if anomalous_test_batch is None:
    print("No anomalous test window found in the current limited test split.")
else:
    anomalous_window_labels = point_labels_to_window_labels(anomalous_test_batch["point_labels"])
    first_anomalous_index = int(torch.where(anomalous_window_labels > 0)[0][0].item())

    plt.figure(figsize=(14, 4))
    plt.plot(anomalous_test_batch["x"][first_anomalous_index, :, :number_of_channels_to_plot].cpu().numpy())
    plt.fill_between(
        np.arange(WINDOW_SIZE),
        anomalous_test_batch["point_labels"][first_anomalous_index].cpu().numpy(),
        alpha=0.25,
        label="point anomaly label",
    )
    plt.title(f"Anomalous test window: {anomalous_test_batch['meta'][first_anomalous_index]}")
    plt.xlabel("window time index")
    plt.legend()
    plt.show()


## 21. FFT Transform Helper

Use FFT magnitude features for frequency-domain baselines or diagnostics. The output shape is `[batch, frequency_bins, channels]`.

In [ ]:
def compute_fft_magnitude(batch_x: torch.Tensor, sampling_rate: float = 1.0) -> tuple[torch.Tensor, torch.Tensor]:
    centered_batch_x = batch_x - batch_x.mean(dim=1, keepdim=True)
    fft_values = torch.fft.rfft(centered_batch_x, dim=1)
    fft_magnitude = torch.abs(fft_values)
    frequency_bins = torch.fft.rfftfreq(centered_batch_x.shape[1], d=1.0 / sampling_rate)
    return frequency_bins, fft_magnitude


In [ ]:
frequency_bins, fft_magnitude = compute_fft_magnitude(sample_batch["x"])

print("frequency bins shape =", tuple(frequency_bins.shape))
print("fft magnitude shape =", tuple(fft_magnitude.shape))


In [ ]:
channel_index = 0

plt.figure(figsize=(10, 4))
plt.plot(frequency_bins.cpu().numpy(), fft_magnitude[0, :, channel_index].cpu().numpy())
plt.title(f"FFT magnitude for channel {channel_index}")
plt.xlabel("frequency bin")
plt.ylabel("magnitude")
plt.show()


## 22. Optional Wavelet Transform Setup

PyWavelets may not be installed in every Colab runtime. If the import fails, uncomment and run the install line, then rerun the import cell.

In [ ]:
try:
    import pywt
except ImportError:
    pywt = None
    print("PyWavelets is not installed. If needed, run: !pip -q install PyWavelets")


In [ ]:
# Run this only when PyWavelets is missing in Colab.
# !pip -q install PyWavelets


## 23. Wavelet Energy Helper

This summarizes each channel by wavelet coefficient energy per decomposition level. The output shape is `[batch, channels, coefficient_groups]`.

In [ ]:
def compute_wavelet_level_energy(
    batch_x: torch.Tensor,
    wavelet_name: str = "db4",
    decomposition_level: int = 3,
) -> torch.Tensor:
    if pywt is None:
        raise ImportError("PyWavelets is required. Run `!pip -q install PyWavelets` in Colab first.")

    batch_array = batch_x.detach().cpu().numpy()
    all_window_energies: list[list[list[float]]] = []

    for window_array in batch_array:
        channel_energies: list[list[float]] = []
        for channel_index in range(window_array.shape[1]):
            coefficients = pywt.wavedec(
                window_array[:, channel_index],
                wavelet=wavelet_name,
                level=decomposition_level,
            )
            coefficient_energies = [float(np.mean(np.square(coefficient))) for coefficient in coefficients]
            channel_energies.append(coefficient_energies)
        all_window_energies.append(channel_energies)

    return torch.tensor(all_window_energies, dtype=torch.float32)


In [ ]:
if pywt is not None:
    wavelet_energy = compute_wavelet_level_energy(sample_batch["x"])
    print("wavelet energy shape =", tuple(wavelet_energy.shape))
else:
    print("Skipping wavelet example because PyWavelets is not installed.")


## 24. Baseline-Friendly Flattening

Classical baselines often expect a 2D array with shape `[batch, features]`. This helper keeps that conversion explicit.

In [ ]:
def flatten_windows_for_baseline(batch_x: torch.Tensor) -> np.ndarray:
    flattened_windows = batch_x.reshape(batch_x.shape[0], -1)
    return flattened_windows.detach().cpu().numpy()


In [ ]:
baseline_x = flatten_windows_for_baseline(sample_batch["x"])
baseline_y = point_labels_to_window_labels(sample_batch["point_labels"]).detach().cpu().numpy()

print("baseline x shape =", baseline_x.shape)
print("baseline y shape =", baseline_y.shape)


## 25. Thesis/Model Adapter Cell

Use this cell as the standard handoff point to your thesis model or any PyTorch baseline. It preserves the batch contract from `documents/design/design_starter.md`.

In [ ]:
def move_batch_to_device(batch: dict[str, Any], device: torch.device) -> dict[str, Any]:
    moved_batch = dict(batch)
    moved_batch["x"] = batch["x"].to(device)
    moved_batch["point_labels"] = batch["point_labels"].to(device) if batch["point_labels"] is not None else None
    moved_batch["mask"] = batch["mask"].to(device) if batch["mask"] is not None else None
    moved_batch["timestamps"] = batch["timestamps"].to(device) if batch["timestamps"] is not None else None
    moved_batch["meta"] = list(batch["meta"])
    return moved_batch


In [ ]:
model_ready_batch = move_batch_to_device(sample_batch, DEVICE)

print("x device =", model_ready_batch["x"].device)
print("x shape =", tuple(model_ready_batch["x"].shape))
print("point_labels shape =", tuple(model_ready_batch["point_labels"].shape))


## 26. Minimal Model Smoke-Test Placeholder

Replace this toy model with your thesis model or a baseline. The important part is that the input is still `batch["x"]` with shape `[B, L, D]`.

In [ ]:
class TinyWindowAutoencoder(torch.nn.Module):
    def __init__(self, number_of_channels: int, hidden_size: int = 64) -> None:
        super().__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(number_of_channels, hidden_size),
            torch.nn.ReLU(),
        )
        self.decoder = torch.nn.Linear(hidden_size, number_of_channels)

    def forward(self, batch: dict[str, Any]) -> dict[str, torch.Tensor]:
        hidden = self.encoder(batch["x"])
        reconstruction = self.decoder(hidden)
        point_scores = torch.mean((reconstruction - batch["x"]) ** 2, dim=-1)
        window_scores = torch.mean(point_scores, dim=-1)
        return {
            "hidden": hidden,
            "recon": reconstruction,
            "point_scores": point_scores,
            "window_scores": window_scores,
        }


In [ ]:
number_of_channels = model_ready_batch["x"].shape[-1]
model = TinyWindowAutoencoder(number_of_channels=number_of_channels).to(DEVICE)

with torch.no_grad():
    outputs = model(model_ready_batch)

print("hidden shape =", tuple(outputs["hidden"].shape))
print("reconstruction shape =", tuple(outputs["recon"].shape))
print("point score shape =", tuple(outputs["point_scores"].shape))
print("window score shape =", tuple(outputs["window_scores"].shape))


## 27. Notes for Reuse

Recommended quick-experiment changes:

- Change `SELECTED_MACHINE_IDS` to another machine or `"all"`.
- Change `WINDOW_SIZE` and `STRIDE` to test window sensitivity.
- Change `MAX_WINDOWS_BY_SPLIT` to trade off speed versus coverage.
- Use `compute_fft_magnitude(...)` for FFT-based diagnostics or baselines.
- Use `compute_wavelet_level_energy(...)` after installing PyWavelets for wavelet-based features.
- Replace `TinyWindowAutoencoder` with your thesis model or another baseline while preserving the same batch contract.
